In [1]:
import os
os.chdir('..')
print(os.getcwd())

/Users/nkapila6/Code/nlpvise/src


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score

def evaluate_multi_label(y_true, y_probs, threshold=0.6):
    y_pred = (y_probs > threshold).astype(int)
    
    per_class_acc = []
    # per_class_mcc = []

    for i in range(y_true.shape[1]):
        per_class_acc.append(accuracy_score(y_true[:, i], y_pred[:, i]))
        # per_class_mcc.append(matthews_corrcoef(y_true[:, i], y_pred[:, i]))

    results = {
        'accuracy': np.mean(per_class_acc),
        # 'mcc': np.mean(per_class_mcc),
        # 'f1': f1_score(y_true, y_pred, average='weighted'),
        # 'auprc': average_precision_score(y_true, y_probs),
        # 'auroc': roc_auc_score(y_true, y_probs)
    }
    # return results
    return np.mean(per_class_acc)

def train_model_torch(model, X_train, y_train, X_test, y_test,
                      lr=0.001, batch_size=128, epochs=8, device='cpu'):
    accuracies = []
    # Move model to device
    model = model.to(device)

    # Data preparation
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test = torch.tensor(y_test, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Training loop
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X).squeeze()
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f}")

        # Evaluation
        with torch.no_grad():
            model.eval()
            # logits = model(X_test).squeeze()
            # probs = torch.sigmoid(logits)
            # preds = (probs > 0.5).float()
            logits = model(X_test)
            probs = torch.sigmoid(logits).cpu().numpy()
            y_test_np = y_test.cpu().numpy()
            acc = evaluate_multi_label(y_test_np, probs)
            accuracies.append(acc)
            print(f"Accuracy {acc}")

    return model, accuracies

In [19]:
import pickle
with open("data/X.pkl", "rb") as f:
            X = pickle.load(f)

with open("data/y.pkl", "rb") as f:
    y = pickle.load(f)

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = ConvNet(input_size=768)  # Replace with your model class

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

trained_model, acc = train_model_torch(
    model=model,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    lr=0.004319,
    batch_size=128,
    epochs=8,
    device=device
)

print(f"✅ Mean Accuracy: {np.mean(acc):.4f} ± {np.std(acc):.4f}")

Epoch 1/8 | Loss: 0.4852
Accuracy 0.7571432804091716
Epoch 2/8 | Loss: 0.4741
Accuracy 0.7657781777644449
Epoch 3/8 | Loss: 0.4708
Accuracy 0.7599561496098278
Epoch 4/8 | Loss: 0.4691
Accuracy 0.7626949472054635
Epoch 5/8 | Loss: 0.4679
Accuracy 0.7610894451666426
Epoch 6/8 | Loss: 0.4673
Accuracy 0.7650263509734861
Epoch 7/8 | Loss: 0.4669
Accuracy 0.7627653152302332
Epoch 8/8 | Loss: 0.4663
Accuracy 0.765980022888126
✅ Mean Accuracy: 0.7626 ± 0.0029
